# Ranking evaluation on the pilot

Results of `evaluate.py` and `judge.py` (dev split): the rankings of the search UI on the benchmark queries with their
reviewed parses, graded 0–3 by the Gemini judge from the main image and the text.

- **random** – reference point; **combined** – query embedding vs description and image;
- **concepts** – residual phrase → nearest concept name → concept scores;
- **bank** – phrase bank, concept scores and text signals, combined breaks ties;
- **bank_heldout** – the same with a bank that lacks the phrases coming only from benchmark queries: its phrases are
  as often unknown as in new queries, so it is the fairer estimate of `bank`;
- **combined_features** – combined plus the embedding of the product's feature text;
- **hybrid_c…_n…** – combined_features plus concept scores of occasion, style and season phrases (weight c) minus
  what a negated phrase excludes (weight n); **hybrid_heldout_…** the same with the held-out bank.

Absolute numbers depend on the pool: every new ranking adds judged products to the ideal order, so compare rankings
within one run of the notebook.

Grades: 3 excellent, 2 good, 1 right product without the soft requirement, 0 wrong. nDCG@10 uses all judged
products of the query; MRR and P@1 count grade ≥ 2 as relevant.

In [ ]:
import sys, json, base64, html
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd
from IPython.display import HTML, Markdown, display

HERE = Path.cwd() if (Path.cwd() / "evaluate.py").exists() else Path.cwd() / "experiments/ranking"
SEARCH_UI = HERE.parents[1] / "search_ui"
sys.path[:0] = [str(HERE), str(SEARCH_UI), str(SEARCH_UI / "core")]
import evaluate as ev
import common

products, meta = common.load_index()
by_id = {p["product_id"]: p for p in products}
scored = set(meta["concepts"]["categories"])
pool = {r["id"]: r for r in map(json.loads, ev.POOL.open(encoding="utf-8")) if r["split"] == "dev"}
runs = [json.loads(line) for line in ev.RUNS.open(encoding="utf-8")]
grades = ev.load_grades()
top = {(r["id"], r["ranking"]): r for r in runs}
RANKINGS = [n for n in ev.RANKINGS if any(k[1] == n for k in top)]
print(f"{len(pool)} queries, {sum(len(g) for g in grades.values())} grades, rankings: {', '.join(RANKINGS)}")

## Metrics per query

In [ ]:
rows = []
for (qid, name), r in top.items():
    q = pool.get(qid)
    if q is None or qid not in grades:
        continue
    m = ev.metrics(r["top"], grades[qid])
    rows.append(dict(id=qid, query=q["query"], category=q["category"], n=q["n"], ranking=name, ms=r["ms"],
                     concept_category=q["category"] in scored,
                     size="2-9" if q["n"] < 10 else "10-49" if q["n"] < 50 else "50+", **m))
pq = pd.DataFrame(rows)
summary = pq.groupby("ranking")[["ndcg", "mrr", "p1", "ms"]].mean().reindex(RANKINGS)
summary["queries"] = pq.groupby("ranking")["ndcg"].count()
summary.style.format({"ndcg": "{:.3f}", "mrr": "{:.3f}", "p1": "{:.3f}", "ms": "{:.0f}"})

### Difference to `combined` (paired over queries, mean and 95% bootstrap interval)

In [ ]:
def diff_table(df, baseline="combined"):
    out = []
    base = df[df.ranking == baseline].set_index("id")
    for name in RANKINGS:
        if name == baseline:
            continue
        cur = df[df.ranking == name].set_index("id")
        row = dict(ranking=name)
        for metric in ("ndcg", "mrr", "p1"):
            d = (cur[metric] - base[metric]).dropna()
            if len(d):
                mean, lo, hi = ev.bootstrap(d.to_numpy())
                row[metric] = f"{mean:+.3f} [{lo:+.3f}, {hi:+.3f}]"
        out.append(row)
    return pd.DataFrame(out).set_index("ranking")

display(diff_table(pq))

### By group: categories with concept scores vs the others, and by number of candidates

In [ ]:
def group_table(df, column, metric="ndcg"):
    t = df.pivot_table(index="ranking", columns=column, values=metric, aggfunc="mean").reindex(RANKINGS)
    counts = df[df.ranking == "combined"].groupby(column)[metric].count()
    t.columns = [f"{c} ({counts.get(c, 0)})" for c in t.columns]
    return t.style.format("{:.3f}")

display(group_table(pq, "concept_category"))
display(group_table(pq, "size"))

In [ ]:
# nDCG by category, combined vs bank vs bank_heldout
HYBRID = "hybrid_c0.5_n1"
by_cat = pq.pivot_table(index="category", columns="ranking", values="ndcg", aggfunc="mean")[
    [n for n in ("combined", "bank", HYBRID, "hybrid_heldout_c0.5_n1") if n in RANKINGS]]
by_cat["queries"] = pq[pq.ranking == "combined"].groupby("category")["ndcg"].count()
by_cat["hybrid - combined"] = by_cat[HYBRID] - by_cat["combined"]
by_cat.sort_values("hybrid - combined").style.format("{:.3f}", subset=by_cat.columns[:-2].tolist() + ["hybrid - combined"])

### `bank_heldout` by how its phrases were read

A query falls into the weakest way any of its phrases was read: unknown < concept name < nearest bank phrase < exact.

In [ ]:
ORDER = ["none", "concept", "nearest", "exact"]
route = {}
for (qid, name), r in top.items():
    if name == "bank_heldout" and r["matches"]:
        route[qid] = min(r["matches"], key=ORDER.index)
pq["heldout_route"] = pq["id"].map(route)
t = pq[pq.heldout_route.notna()].pivot_table(index="ranking", columns="heldout_route", values="ndcg", aggfunc="mean")
t = t.reindex(RANKINGS)[[c for c in ORDER if c in t.columns]]
counts = pd.Series(route).value_counts()
t.columns = [f"{c} ({counts.get(c, 0)})" for c in t.columns]
t.style.format("{:.3f}")

## Browse queries

`show(query_or_id)` draws the top 10 of every ranking with the judge's grade on each image (green 3, light green 2, orange 1, red 0, grey ungraded) and the concept scores the bank used.

In [ ]:
COLORS = {3: "#15803d", 2: "#65a30d", 1: "#d97706", 0: "#b91c1c", None: "#78716c"}
_images = {}

def thumb(pid):
    if pid not in _images:
        p = by_id[pid]
        path = next((x for x in p["images"] if "id_gridfs_1." in x), p["images"][0] if p["images"] else None)
        _images[pid] = base64.b64encode((common.DATASET / path).read_bytes()).decode() if path else ""
    return _images[pid]

def find(query_or_id):
    if query_or_id in pool:
        return query_or_id
    hits = [qid for qid, q in pool.items() if query_or_id.lower() in q["query"].lower()]
    if not hits:
        raise KeyError(query_or_id)
    return hits[0]

def show(query_or_id, rankings=("combined", "bank", "hybrid_c0.5_n1", "hybrid_heldout_c0.5_n1"), k=10):
    qid = find(query_or_id)
    q, g = pool[qid], grades.get(qid, {})
    m = {name: ev.metrics(top[qid, name]["top"], g) for name in rankings if (qid, name) in top}
    out = [f"<h3>{html.escape(q['query'])}</h3><div>{html.escape(q['category'])} · {q['n']} candidates · residual: "
           f"<b>{html.escape(' · '.join(q['residual']))}</b> · id {qid}</div>"]
    for name in rankings:
        if (qid, name) not in top:
            continue
        r, mm = top[qid, name], m[name]
        out.append(f"<div style='margin-top:10px'><b>{name}</b> nDCG {mm['ndcg'] if mm['ndcg'] is None else round(mm['ndcg'], 3)}"
                   f" · P@1 {mm['p1']} · phrases read as: {', '.join(r['matches']) or '–'}</div><div style='display:flex;gap:4px'>")
        for pid in r["top"][:k]:
            p, grade = by_id[pid], g.get(pid)
            concepts = " ".join(f"{c.removeprefix('concept__')}={int(v)}" for c, v in p.items()
                                if c.startswith("concept__") and v == v and v >= 7)[:60]
            out.append(f"<div style='width:110px;font-size:10px'><div style='position:relative'>"
                       f"<img src='data:image/png;base64,{thumb(pid)}' style='width:110px;height:110px;object-fit:contain;background:#fff'>"
                       f"<span style='position:absolute;top:2px;left:2px;background:{COLORS[grade]};color:#fff;padding:0 4px;"
                       f"border-radius:3px;font-weight:bold'>{'?' if grade is None else grade}</span></div>"
                       f"{html.escape(p['name'][:40])}<br><span style='color:#78716c'>{html.escape(concepts)}</span></div>")
        out.append("</div>")
    display(HTML("".join(out)))

### Where the hybrid ranking wins and loses most against `combined`

In [ ]:
wide = pq.pivot_table(index=["id", "query", "category"], columns="ranking", values="ndcg").reset_index()
wide["delta"] = wide["hybrid_c0.5_n1"] - wide["combined"]
cols = ["query", "category", "combined", "bank", "hybrid_c0.5_n1", "hybrid_heldout_c0.5_n1", "delta"]
display(wide.sort_values("delta", ascending=False)[cols].head(10).style.format("{:.3f}", subset=cols[2:]))
display(wide.sort_values("delta")[cols].head(10).style.format("{:.3f}", subset=cols[2:]))

In [ ]:
best, worst = wide.sort_values("delta").iloc[-1]["id"], wide.sort_values("delta").iloc[0]["id"]
show(best)

In [ ]:
show(worst)

In [ ]:
# any query by text or id, e.g. show("funeral")
show(pool[next(iter(pool))]["query"])

## Judge

In [ ]:
dist = pd.Series([g for gs in grades.values() for g in gs.values()]).value_counts().sort_index()
log = [json.loads(line) for line in ev.JUDGMENTS.open(encoding="utf-8")]
print("grades:", dist.to_dict(), f"· calls {len(log)}, errors {sum('error' in r for r in log)}, "
      f"cost ${sum(r.get('cost_usd', 0) for r in log):.2f}")